#Домашнє завдання до тем apply(), groupby(), pivot_table()

В цьому домашньому завданні продовжуємо працювати з набором даних `supermarket_sales.csv`.

0. Імпортуйте бібліотеку pandas та зчитайте дані у змінну `df` типу `pandas.DataFrame`.

In [2]:
import pandas as pd

df = pd.read_csv("supermarket_sales.csv")

1. Дослідимо, який філіал супермаркету ('Branch') є найприбутковішим. Для цього знайдіть сумарний прибуток за кожним філіалом і виявіть, який філіал має найвищий.

In [4]:
branch_profit = df.groupby('Branch')['gross income'].sum()

print(branch_profit)
print("\nНайприбутковіший філіал:", branch_profit.idxmax())

Branch
A    5057.1605
B    5057.0320
C    5265.1765
Name: gross income, dtype: float64

Найприбутковіший філіал: C


2. В якому місті знайходиться філіал з найвищим прибутком? Може в тому місці нам розмітисти ще один магазин.  
Знайдіть відповідь за допомогою функціоналу Pandas.

In [5]:
city_branch_profit = df.groupby(['City', 'Branch'])['gross income'].sum()
print(city_branch_profit)

City       Branch
Mandalay   B         5057.0320
Naypyitaw  C         5265.1765
Yangon     A         5057.1605
Name: gross income, dtype: float64


3.1. Створіть зводну таблицю, яка покаже, скільки покупок (інвойсів) було зроблено в кожній з філій (`Branch`) за різними категоріями товарів. Запишіть таблицю в змінну `invoices_by_category` і виведіть змінну на екран.
Ця таблиця допоможе проаналізувати, в якій філії купують найбільше товарів кожної з категорій.

In [8]:
invoices_by_category = df.pivot_table(
    index='Branch',
    columns='Product line',
    values='Invoice ID',
    aggfunc='count'
)

invoices_by_category

Product line,Electronic accessories,Fashion accessories,Food and beverages,Health and beauty,Home and lifestyle,Sports and travel
Branch,,,,,,
A,60,51,58,47,65,59
B,55,62,50,53,50,62
C,55,65,66,52,45,45


Очікуваний результат:

![](https://drive.google.com/uc?export=view&id=1rueAdko6S3UxIHGtojetTxlES-EyM6Yb)

3.2. Викристовуючи змінну `invoices_by_category` дайте відповідь програмно (тобто значення треба не просто знайти очима, а вивести за допомогою коду), в якому філіалі магазину (`Branch`) найбільше інвойсів із покупкою товарів категорії "Електронні аксесуари" (`Electronic accessories`)?


In [10]:
top_branch_electronic = invoices_by_category['Electronic accessories'].idxmax()

print(f"Філіал з найбільшою кількістю інвойсів у категорії 'Electronic accessories': {top_branch_electronic}")

Філіал з найбільшою кількістю інвойсів у категорії 'Electronic accessories': A


4-6. **Творче завдання на розвиток аналітичного мислення**

Крок 1. Сформулюйте ТРИ питання (гіпотези) до наявних даних, які допомогли б вам зрозуміти, які користувачі що, де та коли найбільше/найменше купують, аби дати на основі цих гіпотез рекомендації бізнесу. Звісно питання мають бути не тими, на які ми вже відповіли в завданнях модулю.

Крок 2. Знайдіть відповіді на свої питання з допомогою функціоналу pandas.

Крок 3. Напишіть, як відповідь на це питання може бути використана для прийняття бізнес рішень.   
   
 Питання можуть бути будь-якої складності, але їх має бути 3. Кожне питання оцінюється як 1 завдання. Без виконання цього завдання ДЗ не приймається. Якщо є питання щодо виконання - пишіть у чат 🙌


Крок 1. Формулювання трьох питань (гіпотез)
Гіпотеза 1 (Спосіб оплати): Чи залежить вибір методу оплати (Payment) від гендеру покупця (Gender), і чи є переважаючий метод у певній категорії покупців?

Гіпотеза 2 (Час доби / Сегментація за годинами): У який період дня (ранок, день, вечір) покупці роблять найбільші за сумою покупки (Total)?

Гіпотеза 3 (Тип покупця та оцінка): Чи відрізняється задоволеність (середній Rating) між звичайними покупцями (Normal) та тими, хто має членство (Member)?



In [14]:
#1
payment_gender = df.pivot_table(index='Gender', columns='Payment', values='Invoice ID', aggfunc='count')
payment_gender

Payment,Cash,Credit card,Ewallet
Gender,,,
Female,178,163,160
Male,166,148,185


In [17]:
# гіпотеза 2
import numpy as np
def get_hour(t):
    return int(t.split(':')[0])

df['Hour'] = df['Time'].apply(get_hour)

# Гіпотеза 2: розподіл за періодами дня
conditions_time = [
    (df['Hour'] >= 6) & (df['Hour'] < 12),
    (df['Hour'] >= 12) & (df['Hour'] < 18),
    (df['Hour'] >= 18)
]
choices_time = ['Morning', 'Afternoon', 'Evening']
df['Time_Period'] = np.select(conditions_time, choices_time, default='Night')

# Рахуємо середню суму чеку в залежності від періоду дня
avg_total_by_period = df.groupby('Time_Period')['Total'].mean()
avg_total_by_period

,Total
Time_Period,
Afternoon,326.644997
Evening,315.656157
Morning,323.553982


In [19]:
#гіпотеза 3
customer_stats = df.groupby('Customer type').agg({'Rating': 'mean', 'Total': ['mean', 'count']})
customer_stats

Rating       Total      
                   mean        mean count
Customer type                            
Member         6.940319  327.791305   501
Normal         7.005210  318.122856   499

Крок 3. Використання результатів для прийняття бізнес-рішень
Щодо способу оплати: Жіноча стать частіше використовує готівку і кредитну карту, маркетинг може налаштувати таргетовані акції або кешбеки спільно з платіжними системами для стимулювання безготівкових платежів, що зменшить витрати на інкасацію.

Щодо часу доби: найбільші чеки припадають на обідній час, супермаркету варто перерозподілити робочий персонал (збільшити кількість касирів у обідні години), щоб уникнути черг і не втратити великі чеки через залишені кошики.

Щодо типу покупця (Member vs Normal): Учасники програми лояльності (Member) здійснюють дещо більше покупок (501 проти 499) і мають вищий середній чек (327,79 проти 318,12), проте їхній рівень задоволеності майже не відрізняється від звичайних клієнтів (Normal) — середній рейтинг становить 6,94 проти 7,01 відповідно, що свідчить про необхідність покращення привілеїв програми лояльності для підвищення їхньої задоволеності.